# Sleep by hour — a step-by-step analysis

This notebook turns a finished ebb-viewer project into a **sleep-by-hour table**: for each
animal, each sleep state (wake / NREM / REM) and each 2-hour clock bin, how much of the time
was spent in that state, how many bouts there were, and how long a bout lasted. It also writes
a whole-recording summary, a Prism workbook and group plots.

**Make a copy of this notebook for each analysis** and edit the Settings cell (and the groups
in section 10). Every number is computed here, in small functions, with a toy example before
each real calculation. The `ebb` library is only used in two optional checks, at the end of
sections 9 and 15.

**What it reads from your project folder**

- `spindle_outputs/<combo>/*.csv`: one CSV per animal; we use the `final_state` column, the
  state after the Reclassify step (NREM gap fill, EMG gate and any SWD review edits). This is
  the same state the Sleep QC plots and the PSDs use.
- `standard/*PROCESSED.edf`: only the header, for the time each recording began.
- `manifest.json`: the project's channel combo and which animals were excluded in Sleep QC.

**The steps**

1. Find the animals, their recording start times, and the excluded animals
2. Read each animal's state per epoch
3. Upsample from one state per 4 s epoch to one state per second
4. Rotate the recording so it starts at 6 AM
5. Score each 2-hour bin: fraction of time, and bouts (found over the whole recording and
   counted in the bin that holds most of each one)
6. Do that for every bin of one animal
7. Repeat for every animal
8. Fold the bins onto a 24-hour day and average across days
9. Check against the library (optional)
10. Assign groups; 11. convert to ZT; 12. lay out a Prism sheet; 13. write the workbook;
    14. plot by group
15. A whole-recording summary

In [ ]:
import json
from datetime import datetime, time, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 0. Settings

Everything you might change lives in this one cell.

- `PROJECT_ROOT` – your project folder (the one that holds `manifest.json`).
- `STATE_COLUMN` – which CSV column holds the state. `final_state` is the reclassified state;
  leave it unless you want to compare with the original SPINDLE states (`rawstate`).
- `EPOCH_SEC` – SPINDLE scores one state per 4-second epoch.
- `BIN_HOURS` – width of each bin.
- `BEGIN_HR` – the clock hour that bin 0 should start at (6 = 6 AM, i.e. lights-on).
- `TZ` – the time zone the recordings were made in, so a daylight-saving change is counted
  correctly when finding the first 6 AM. Use `None` to ignore daylight saving.
- `INCLUDE_EXCLUDED` – `False` leaves out animals excluded in Sleep QC; `True` keeps them and
  marks them in an `excluded` column.

In [ ]:
PROJECT_ROOT = Path("/path/to/your/project")

manifest = json.loads((PROJECT_ROOT / "manifest.json").read_text())
COMBO = manifest["combo"]                          # e.g. "LFC_RSC"
SPINDLE_DIR = PROJECT_ROOT / "spindle_outputs" / COMBO
STANDARD_DIR = PROJECT_ROOT / "standard"

STATE_COLUMN = "final_state"
EPOCH_SEC = 4
BIN_HOURS = 2
BEGIN_HR = 6
TZ = ZoneInfo("America/Chicago")
INCLUDE_EXCLUDED = False

STATES = ["w", "n", "r"]
STATE_NAMES = {"w": "Wake", "n": "NREM", "r": "REM"}

## 1. Animals, recording start times and exclusions

**Animal ID.** Each CSV name starts with the animal ID, followed by an underscore:
`Mice-1248_KO_16_54_..._labels.csv` belongs to `Mice-1248`. The ID is everything before the
first `_`.

**Start times.** A CSV is just a list of states; it doesn't say *when* the recording began.
We need that to know which epochs fall at 6 AM. The start time is stored in the header of
each preprocessed EDF: the first 256 bytes of an EDF are a fixed-layout text header, with the
start date (`dd.mm.yy`) at bytes 168-175 and the start time (`hh.mm.ss`) at bytes 176-183. We
read just those bytes, so no EDF library is needed.

In [ ]:
def animal_id_from_path(path: Path) -> str:
    return path.name.split("_")[0]


def edf_start_datetime(path: Path) -> datetime:
    """Recording start time from the EDF header (bytes 168-175 date, 176-183 time)."""
    with open(path, "rb") as f:
        header = f.read(256)
    date, clock = header[168:176].decode("ascii"), header[176:184].decode("ascii")
    return datetime.strptime(f"{date} {clock}", "%d.%m.%y %H.%M.%S")


csv_paths = sorted(SPINDLE_DIR.glob("*.csv"))
ids = [animal_id_from_path(p) for p in csv_paths]
assert ids, f"no CSVs in {SPINDLE_DIR}"
assert len(ids) == len(set(ids)), "two CSVs for the same animal in this folder"

edf_by_animal = {animal_id_from_path(p): p for p in STANDARD_DIR.glob("*PROCESSED.edf")}
missing = [a for a in ids if a not in edf_by_animal]
assert not missing, f"no EDF in {STANDARD_DIR} for: {missing}"

start_by_animal = {a: edf_start_datetime(edf_by_animal[a]) for a in ids}
print(f"{len(csv_paths)} CSVs, all with a start time")
pd.Series(start_by_animal, name="start").head()

**Excluded animals.** In ebb-viewer an animal can be excluded in Sleep QC (or in PSD review)
with a reason. That is recorded in `manifest.json`: `sleep_qc` is `"excluded"` and the reason
is in the animal's `notes`. By default we leave those animals out of the analysis.

We also warn about animals whose Sleep QC was never decided (`ok` or `excluded`).

In [ ]:
animals_in_manifest = manifest["animals"]
excluded = {a: e["notes"] for a, e in animals_in_manifest.items() if e["sleep_qc"] == "excluded"}
undecided = [a for a in ids if animals_in_manifest.get(a, {}).get("sleep_qc") not in ("ok", "excluded")]

if undecided:
    print("WARNING: Sleep QC not decided for", undecided)
print(f"{len(excluded)} excluded animals:")
for animal, why in excluded.items():
    print(f"  {animal}: {why.strip().replace(chr(10), '; ')}")

if not INCLUDE_EXCLUDED:
    csv_paths = [p for p in csv_paths if animal_id_from_path(p) not in excluded]
    ids = [animal_id_from_path(p) for p in csv_paths]
print(f"analysing {len(ids)} animals")

## 2. Read one animal's states

We'll work through one animal in detail, then loop over everyone.

SPINDLE's CSV has one row per 4-second epoch. The state we want is a single letter
(`w` wake, `n` NREM, `r` REM). The column `final_state` holds the state after reclassification,
which is what every other step of ebb-viewer uses. (`rawstate` / `state` hold SPINDLE's original
scoring, before gap filling, the EMG gate and SWD review.) If a CSV has no `final_state` yet,
the Reclassify step hasn't been run on it.

In [ ]:
def load_states(path: Path) -> np.ndarray:
    header = pd.read_csv(path, nrows=0).columns
    if STATE_COLUMN not in header:
        raise ValueError(f"{path.name} has no '{STATE_COLUMN}' column. Run the Reclassify step first.")
    states = pd.read_csv(path, usecols=[STATE_COLUMN])[STATE_COLUMN].to_numpy()
    assert set(states) <= set(STATES), f"unexpected state values: {set(states) - set(STATES)}"
    return states


example_path = csv_paths[0]
example_id = animal_id_from_path(example_path)
epoch_states = load_states(example_path)

print(example_id)
print("epochs:", len(epoch_states))
print("hours :", len(epoch_states) * EPOCH_SEC / 3600)
print("first 20 epochs:", "".join(epoch_states[:20]))

Quick look at the raw hypnogram (state over time), measured from the start of the recording.

In [ ]:
def plot_hypnogram(states, step_sec, t0_hr=0, title=""):
    level = {"r": 0, "n": 1, "w": 2}
    y = np.array([level[s] for s in states])
    t = t0_hr + np.arange(len(y)) * step_sec / 3600
    fig, ax = plt.subplots(figsize=(11, 2.2))
    ax.step(t, y, where="post", lw=0.6)
    ax.set_yticks([0, 1, 2], ["REM", "NREM", "Wake"])
    ax.set_xlabel("hours")
    ax.set_title(title)
    return ax


plot_hypnogram(epoch_states, EPOCH_SEC, title=f"{example_id}: raw epochs, from recording start");

## 3. Upsample to one state per second

Every later step is easier if one array position equals one second. Each epoch is 4 s long,
so we repeat each state 4 times.

Toy example: three epochs `w n r` become twelve seconds.

In [ ]:
toy = np.array(["w", "n", "r"])
print(np.repeat(toy, EPOCH_SEC))

In [ ]:
def upsample(epoch_states: np.ndarray, epoch_sec: int = EPOCH_SEC) -> np.ndarray:
    return np.repeat(epoch_states, epoch_sec)


sec_states = upsample(epoch_states)
print(f"{len(epoch_states)} epochs -> {len(sec_states)} seconds")

## 4. Rotate so the recording starts at 6 AM

Animals don't all start recording at the same time of day, so their raw bin 0 means different
clock times. To compare animals, we shift each recording so that **its first 6 AM is
position 0**, and the stretch that came *before* that first 6 AM moves to the end.

1. Find how many seconds pass between the recording start and the next 6:00.
2. Drop that many seconds from the front of the array and tack them onto the back.

Example: a recording starting at 17:56:19 reaches the next 6 AM after 12 h 3 min 41 s
= 43,421 s.

If the night of that first 6 AM contains a daylight-saving change, the clock hour is 23 or 25
real hours long. With `TZ` set we count *real elapsed seconds* by converting both times to UTC
before subtracting. (Python skips this conversion when two times share a time zone, which would
give the plain clock difference, so we ask for UTC explicitly.)

In [ ]:
def seconds_until_next(start: datetime, hour: int, tz=TZ) -> int:
    """Real seconds from `start` forward to the next `hour`:00 (today's if not yet passed)."""
    if tz is not None:
        start = start.replace(tzinfo=tz)
    target = start.replace(hour=hour, minute=0, second=0, microsecond=0)
    if start > target:
        target = datetime.combine((start + timedelta(days=1)).date(), time(hour), tzinfo=start.tzinfo)
    if tz is not None:
        return int((target.astimezone(timezone.utc) - start.astimezone(timezone.utc)).total_seconds())
    return int((target - start).total_seconds())


example_start = start_by_animal[example_id]
offset = seconds_until_next(example_start, BEGIN_HR)
print("recording starts:", example_start)
print("seconds until next 6 AM:", offset, f"({offset / 3600:.2f} h)")

In [ ]:
def rotate(sec_states: np.ndarray, offset: int) -> np.ndarray:
    return np.concatenate([sec_states[offset:], sec_states[:offset]])


rotated = rotate(sec_states, offset)

# the same thing, spelled out: rotated[0] is the state at exactly 6 AM
assert rotated[0] == sec_states[offset]
assert len(rotated) == len(sec_states)

Two things to keep in mind:

- A recording shorter than `offset` never reaches 6 AM, so it can't be rotated. Such animals
  are skipped (with a printed warning) below.
- The rotation counts real elapsed time, so the first 6 AM is exact. After that the recording
  is cut into fixed 2-hour pieces of real time. If it crosses a daylight-saving change, the bins
  after the change are an hour off the clock (the library does the same).

## 5. Score a bin: fraction of time and bouts

Cut the rotated array into consecutive 2-hour pieces (7,200 seconds each). For every state in
a piece we compute three numbers.

- **`frac_time`** – seconds in the state ÷ seconds in the bin.
- **`n_bouts`** – number of bouts of that state counted in the bin. A *bout* is an unbroken
  stretch of the state.
- **`mean_bout_sec`** – the average full length of those bouts in seconds (`NaN` if there were none).

Time in the state is measured inside the bin. Bouts are different: a bout doesn't care where we
cut the bins. So we find the bouts first, over the **whole recording**, and then decide which bin
each one belongs to. A bout that crosses a bin edge is counted **once**, in the bin that holds
most of it, and keeps its full length. (Cutting the recording first and counting bin by bin would
split such a bout in two, count it twice, and shorten both pieces.)

Finding bouts is the tricky part, so here it is on a toy example. Take a mask that is `True`
wherever the state is present:

In [ ]:
mask = np.array([0, 1, 1, 0, 1, 0, 0, 1, 1, 1], dtype=bool)

change = np.diff(mask.astype(int))    # +1 where a run starts, -1 where it ends
starts = np.where(change == 1)[0] + 1   # the run begins on the sample *after* the change
ends = np.where(change == -1)[0] + 1    # one past the last sample of the run

print("diff  :", change)
print("starts:", starts, "  ends:", ends)

The last run (positions 7–9) has a start but no end, because the array ends first. Likewise a
run that is already going at position 0 has an end but no start. The simple fix is to put a
`False` at both ends of the mask before looking for changes, so every run has a start and an end:

In [ ]:
def bout_runs(mask: np.ndarray):
    """Start (inclusive) and end (exclusive) of every bout: an unbroken run of True in `mask`."""
    change = np.diff(np.r_[False, mask, False].astype(int))
    return np.where(change == 1)[0], np.where(change == -1)[0]


def bouts(mask: np.ndarray):
    """Return (number of bouts, mean bout length in samples) for a boolean mask."""
    starts, ends = bout_runs(mask)
    if len(starts) == 0:
        return 0, np.nan
    return len(starts), (ends - starts).mean()


n, mean_len = bouts(mask)
print(f"{n} bouts, mean length {mean_len:.2f} samples")   # runs of length 2, 1, 3

Now the question of **which bin** a bout belongs to. Take 30 seconds cut into three bins of
10 seconds, and a bout that runs from second 6 to second 14. Four of its seconds are in bin 0 and
five are in bin 1:

In [ ]:
toy_mask = np.zeros(30, dtype=bool)
toy_mask[6:15] = True                       # one bout, 9 seconds long
toy_bin_of_second = np.arange(30) // 10     # the bin each second falls in: 0 for 0-9, 1 for 10-19, ...

starts, ends = bout_runs(toy_mask)
print("the bout covers seconds", starts[0], "to", ends[0] - 1)
print("seconds of it in each bin:", np.bincount(toy_bin_of_second[starts[0]:ends[0]], minlength=3))

The bout goes to the bin with the most of its seconds (bin 1 here). If two bins hold the same
number of seconds, the earlier bin wins. The function below does this for every bout of a state:
it counts the bouts in each bin and averages their *full* lengths.

In [ ]:
def bouts_by_bin(mask: np.ndarray, bin_of_second: np.ndarray, n_bins: int):
    """Whole bouts per bin: each bout is counted once, in the bin that holds most of it.

    Returns (number of bouts per bin, mean bout length in seconds per bin; NaN if none).
    """
    starts, ends = bout_runs(mask)
    if len(starts) == 0:
        return np.zeros(n_bins, dtype=int), np.full(n_bins, np.nan)
    bin_of_bout = bin_of_second[starts].copy()                              # the bin each bout starts in
    for k in np.where(bin_of_second[ends - 1] != bin_of_bout)[0]:           # bouts that cross a bin edge
        bin_of_bout[k] = np.bincount(bin_of_second[starts[k]:ends[k]]).argmax()
    counts = np.bincount(bin_of_bout, minlength=n_bins)
    totals = np.bincount(bin_of_bout, weights=ends - starts, minlength=n_bins)
    return counts, np.where(counts > 0, totals / np.maximum(counts, 1), np.nan)


counts, mean_sec = bouts_by_bin(toy_mask, toy_bin_of_second, 3)
print("bouts per bin:", counts)               # [0 1 0]
print("mean length  :", mean_sec)             # [nan  9. nan]

Now one real bin by hand: the first 2 hours of the rotated example animal (so 6–8 AM).

Time in state is read from the rotated array. Bouts are found on the **original** array
(`sec_states`), and each is then placed in a bin. We use the original order because rotating
moves the hours before the first 6 AM to the end: searching the rotated array would join two
stretches that were never next to each other. A bout that crosses the 6 AM rotation point is
still one bout; it simply goes to the bin that holds most of it.

In [ ]:
bin_len = BIN_HOURS * 3600
n_bins = -(-len(sec_states) // bin_len)
# the bin each second of the ORIGINAL recording falls in once it is rotated to start at 6 AM
bin_of_second = ((np.arange(len(sec_states)) - offset) % len(sec_states)) // bin_len

first_bin = rotated[:bin_len]

rows = []
for state in STATES:
    m = first_bin == state
    counts, mean_sec = bouts_by_bin(sec_states == state, bin_of_second, n_bins)
    rows.append({"state": state, "frac_time": m.sum() / len(first_bin),
                 "n_bouts": counts[0], "mean_bout_sec": mean_sec[0]})
pd.DataFrame(rows)

Fractions across the three states should add to 1.

In [ ]:
assert abs(sum(r["frac_time"] for r in rows) - 1) < 1e-9

## 6. All bins for one animal

Loop over every bin. (The bouts of each state are found once for the whole recording, not once per bin.) The last bin is usually shorter than 2 hours (the recording rarely
divides evenly), so its fractions are taken over *its own* length.

`bin_start_hr` is the clock hour the bin begins: bin 0 starts at `BEGIN_HR`, and it grows by
`BIN_HOURS` each bin, *without* wrapping at 24. A 72 h recording therefore has bin starts
6, 8, … 28, 30, … so day 2 and day 3 stay distinct for now.

In [ ]:
def bin_metrics(sec_states: np.ndarray, offset: int, animal: str) -> pd.DataFrame:
    bin_len = BIN_HOURS * 3600
    n_bins = -(-len(sec_states) // bin_len)
    rotated = rotate(sec_states, offset)
    bin_of_second = ((np.arange(len(sec_states)) - offset) % len(sec_states)) // bin_len
    bouts_in = {state: bouts_by_bin(sec_states == state, bin_of_second, n_bins) for state in STATES}
    rows = []
    for bin_idx, start in enumerate(range(0, len(rotated), bin_len)):
        chunk = rotated[start:start + bin_len]
        for state in STATES:
            counts, mean_sec = bouts_in[state]
            rows.append({
                "mouse_id": animal,
                "bin_idx": bin_idx,
                "bin_start_hr": BEGIN_HR + bin_idx * BIN_HOURS,
                "state": state,
                "frac_time": (chunk == state).sum() / len(chunk),
                "n_bouts": int(counts[bin_idx]),
                "mean_bout_sec": mean_sec[bin_idx],
            })
    return pd.DataFrame(rows)


example_bins = bin_metrics(sec_states, offset, example_id)
example_bins.head(6)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
for state in STATES:
    sub = example_bins[example_bins["state"] == state]
    ax.plot(sub["bin_start_hr"], sub["frac_time"], marker="o", label=STATE_NAMES[state])
ax.set_xlabel("hours since midnight of day 1 (6 = first 6 AM)")
ax.set_ylabel("fraction of bin")
ax.set_title(f"{example_id}: fraction of time per 2-hour bin")
ax.legend();

## 7. Every animal

Now we wrap sections 2–6 into one function and run it over all the CSVs. Animals whose
recording is too short to reach 6 AM are skipped with a message.

In [ ]:
def sleep_bins_for_animal(path: Path):
    animal = animal_id_from_path(path)
    sec_states = upsample(load_states(path))
    offset = seconds_until_next(start_by_animal[animal], BEGIN_HR)
    if offset >= len(sec_states):
        print(f"skipping {animal}: recording ends before it reaches {BEGIN_HR}:00")
        return None
    return bin_metrics(sec_states, offset, animal)


per_animal = [sleep_bins_for_animal(p) for p in csv_paths]
sleep_bins = pd.concat([d for d in per_animal if d is not None], ignore_index=True)

print(sleep_bins["mouse_id"].nunique(), "animals,", len(sleep_bins), "rows")
sleep_bins.head()

## 8. Fold onto a 24-hour day and average across days

`sleep_bins` has one row per animal × state × bin, with bins running across all days of the
recording. To get a typical day, we map each bin onto its clock hour with `% 24` (hour 30 is
6 AM again), then average each metric over the days that share that hour.

`n_days` records how many bins went into each average. It's lower than the number of days
for any hour that the recording only partly covers. Note that every bin counts equally, even a
short final one.

In [ ]:
METRICS = ["frac_time", "n_bouts", "mean_bout_sec"]

sleep_bins["hour_of_day"] = (sleep_bins["bin_start_hr"] % 24).astype(int)

grouped = sleep_bins.groupby(["mouse_id", "state", "hour_of_day"], as_index=False)
sleep_bins_avg = grouped[METRICS].mean()
sleep_bins_avg["n_days"] = grouped.size()["size"]
sleep_bins_avg = sleep_bins_avg.sort_values(["mouse_id", "state", "hour_of_day"], ignore_index=True)

sleep_bins_avg.head(12)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharex=True)
for ax, state in zip(axes, STATES):
    sub = sleep_bins_avg[sleep_bins_avg["state"] == state]
    for animal, d in sub.groupby("mouse_id"):
        ax.plot(d["hour_of_day"], d["frac_time"], lw=0.8, alpha=0.6)
    ax.set_title(STATE_NAMES[state])
    ax.set_xlabel("clock hour (bin start)")
axes[0].set_ylabel("fraction of time")
fig.suptitle("Average day, one line per animal");

## 9. Check against the library (optional)

This is the first of two places that use `ebb`. It runs the pipeline's own
calculation on the same CSVs and EDFs and compares. Skip it if you don't have `ebb` installed.

The library also reads `final_state`, takes start times from the EDF headers and counts
daylight saving the same way, so the numbers should be identical. It includes every animal in the
folder, so we compare only the rows computed above.

In [ ]:
from ebb.process.sleep_compute import compute_sleep_bins_df_offset

lib = compute_sleep_bins_df_offset(
    SPINDLE_DIR, STANDARD_DIR, bin_hours=BIN_HOURS, begin_hr=BEGIN_HR, tz=TZ
)

key = ["mouse_id", "bin_idx", "state"]
merged = sleep_bins.merge(lib, on=key, suffixes=("", "_lib"), how="inner")
print(f"{len(merged)} of {len(sleep_bins)} rows matched on {key}")

for col in METRICS:
    # equal_nan: mean_bout_sec is NaN when a state has no bouts in the bin
    ok = np.allclose(merged[col], merged[f"{col}_lib"], equal_nan=True)
    print(f"{col:15s} matches library: {ok}")

## 10. Assign animals to groups

Prism wants animals blocked by experimental group. Edit `groups` below: keys are group names,
values are lists of animal IDs exactly as printed (the part of the filename before the first
`_`, e.g. `Mice-1248` or `Mice-C266`). Leave `groups` empty to treat everyone as one group.
Instead of typing them, you can read the groups from a CSV file: set `GROUPS_CSV` to its path.
The easiest layout is one row per animal with two columns named `animal` and `group`; one column
per group (the group name on top, its animal IDs below) also works.
The assertions catch an animal you forgot, or listed twice.

In [ ]:
import csv


def read_groups_csv(path):
    """``{group: [animal IDs]}`` from a CSV, in either of two layouts.

    * One row per animal, with columns named ``animal`` and ``group`` (other columns are ignored).
    * One column per group: the group name in the first row, its animal IDs below it.

    Blank cells and rows are skipped, and a group with no animals is dropped. An animal listed
    in two groups is an error.
    """
    with open(path, newline="", encoding="utf-8-sig") as f:
        lines = [
            (n, [cell.strip() for cell in row]) for n, row in enumerate(csv.reader(f), start=1)
        ]
    lines = [(n, row) for n, row in lines if any(row)]
    if not lines:
        raise ValueError(f"{path} is empty")
    header, body = lines[0][1], lines[1:]
    lowered = [name.lower() for name in header]

    groups = {}
    if "animal" in lowered and "group" in lowered:
        i_animal, i_group = lowered.index("animal"), lowered.index("group")
        for n, row in body:
            animal = row[i_animal] if i_animal < len(row) else ""
            group = row[i_group] if i_group < len(row) else ""
            if animal and group:
                groups.setdefault(group, []).append(animal)
            elif animal or group:
                raise ValueError(f"{path}, line {n}: a row needs both an animal and a group")
    else:
        for col, name in enumerate(header):
            animals = [row[col] for _, row in body if col < len(row) and row[col]]
            if animals and not name:
                raise ValueError(
                    f"{path}: column {col + 1} has animals but no group name in its first row"
                )
            if name:
                groups[name] = animals

    groups = {group: list(dict.fromkeys(animals)) for group, animals in groups.items() if animals}
    owner = {}
    for group, animals in groups.items():
        for animal in animals:
            if owner.setdefault(animal, group) != group:
                raise ValueError(
                    f"{path}: {animal} is in more than one group ({owner[animal]}, {group})"
                )
    return groups


print("animals:", ", ".join(sorted(ids)))

GROUPS_CSV = None           # e.g. Path("groups.csv"); if set, replaces the groups typed below

groups = {
    # "WT": ["Mice-1248", "Mice-1253"],
    # "MUT": ["Mice-1251", "Mice-1279"],
}
if GROUPS_CSV:
    groups = read_groups_csv(GROUPS_CSV)
if not groups:
    groups = {"All": sorted(ids)}

animal_to_group = {a: g for g, animals in groups.items() for a in animals}
assert len(animal_to_group) == sum(len(v) for v in groups.values()), "an animal is in two groups"

sleep_bins_avg["group"] = sleep_bins_avg["mouse_id"].map(animal_to_group)
sleep_bins_avg["excluded"] = sleep_bins_avg["mouse_id"].isin(excluded)
unassigned = sorted(sleep_bins_avg.loc[sleep_bins_avg["group"].isna(), "mouse_id"].unique())
assert not unassigned, f"not in any group: {unassigned}"
not_analysed = sorted(set(animal_to_group) - set(ids))
if not_analysed:
    print("listed in groups but not analysed (excluded or no CSV):", not_analysed)
print({g: len(a) for g, a in groups.items()})

## 11. Clock hour to ZT

*Zeitgeber time* counts hours from lights-on, so ZT 0 is the lights-on hour and lights-off is
ZT 14 here (8 PM, a 14:10 light cycle). With lights on at 6 AM, ZT = (clock hour − 6) mod 24. The mod
keeps the result in 0–23: clock hour 4 becomes ZT 22.

In [ ]:
ZT_ANCHOR_HOUR = 6   # clock hour lights come on

# label used on plots and sheet headers, e.g. "ZT hours (0 = 6 AM)"
ZT_LABEL = f"ZT hours (0 = {ZT_ANCHOR_HOUR % 12 or 12} {'AM' if ZT_ANCHOR_HOUR % 24 < 12 else 'PM'})"

sleep_bins_avg["zt"] = (sleep_bins_avg["hour_of_day"] - ZT_ANCHOR_HOUR) % 24
sleep_bins_avg[["hour_of_day", "zt"]].drop_duplicates().sort_values("zt").T

## 12. Lay out one Prism sheet

A Prism grouped table wants **one row per time bin and one column per animal**, with a header
row naming the group above each animal. We make one such table for each *state × metric*.

First, a single one by hand: wake `frac_time`. `pivot` turns the long table (one row per
animal × bin) into rows = ZT, columns = animals.

In [ ]:
sub = sleep_bins_avg[sleep_bins_avg["state"] == "w"]
wide = sub.pivot(index="zt", columns="mouse_id", values="frac_time")

# order the columns by group (in the order of `groups`), then by animal ID
column_order = [a for g in groups for a in sorted(groups[g]) if a in wide.columns]
wide = wide[column_order]
wide.head()

Prism reads two header rows: group names, then animal IDs. Putting both in a `MultiIndex`
puts the group name above each animal.

In [ ]:
wide.columns = pd.MultiIndex.from_tuples(
    [(animal_to_group[a], a) for a in wide.columns], names=["group", "animal"]
)
wide.index = [f"{zt}-{zt + BIN_HOURS}" for zt in wide.index]   # row labels like '0-2'
wide.index.name = ZT_LABEL
wide.head()

Groups with fewer animals than the largest group get blank columns at the end of their block
when the real workbook is written, so every group block is the same width in Prism.

Now the same layout for every state × metric, written to an Excel workbook with one sheet each
(`w_frac_time`, `w_bouts`, `w_mean_bout_sec`, …).

In [ ]:
from openpyxl import Workbook

def prism_sheet_frame(df, state, metric):
    """Rows = ZT bins, columns = animals (grouped), values = `metric` for `state`."""
    wide = df[df["state"] == state].pivot(index="zt", columns="mouse_id", values=metric)
    return wide.reindex(sorted(wide.index))


SHEET_METRIC = {"n_bouts": "bouts"}   # sheet names: w_frac_time, w_bouts, w_mean_bout_sec, ...


def write_prism_workbook(df, out_path, groups, metrics, states=STATES):
    # pad every group to the largest group's size with blank columns
    width = max(len(a) for a in groups.values())
    wb = Workbook()
    wb.remove(wb.active)

    for metric in metrics:
        for state in states:
            wide = prism_sheet_frame(df, state, metric)
            ws = wb.create_sheet(f"{state}_{SHEET_METRIC.get(metric, metric)}")
            ws.cell(row=2, column=1, value=ZT_LABEL)

            col = 2
            layout = []                                          # animal (or None) per column
            for g, animals in groups.items():
                block = sorted(a for a in animals if a in wide.columns)
                block += [None] * (width - len(block)) if block else []
                for animal in block:
                    ws.cell(row=1, column=col, value=g)
                    ws.cell(row=2, column=col, value=animal)
                    layout.append(animal)
                    col += 1

            for r, zt in enumerate(wide.index, start=3):
                ws.cell(row=r, column=1, value=f"{zt}-{zt + BIN_HOURS}")
                for c, animal in enumerate(layout, start=2):
                    val = wide.at[zt, animal] if animal in wide.columns else None
                    ws.cell(row=r, column=c, value=None if pd.isna(val) else float(val))
            ws.freeze_panes = "B3"
    wb.save(out_path)
    return out_path

## 13. Write the workbook

`n_bouts` is a count per bin and `mean_bout_sec` is in seconds; `frac_time` is a fraction of
the bin (0–1), so multiply by 100 in Prism for percent.

In [ ]:
OUT_DIR = PROJECT_ROOT / "sleep_analysis"
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "prism_sleep_by_hour.xlsx"
write_prism_workbook(sleep_bins_avg, OUT_PATH, groups, metrics=METRICS)
sleep_bins.to_csv(OUT_DIR / "sleep_bins.csv", index=False)
sleep_bins_avg.to_csv(OUT_DIR / "sleep_bins_avg.csv", index=False)
print("Wrote", OUT_PATH)

check = pd.read_excel(OUT_PATH, sheet_name=None, header=None)
print(list(check))
check["w_frac_time"].iloc[:6, :5]

## 14. Plot sleep architecture by group

Group mean ± SEM at each ZT bin. Rows are the three metrics, columns the three states; the
grey band is lights-off (ZT 14–24).

In [ ]:
DARK_ONSET_ZT = 14
ROWS = [("n_bouts", "Bout number", "bouts per bin", 1),
        ("mean_bout_sec", "Bout duration", "seconds", 1),
        ("frac_time", "Time in state", "% of bin", 100)]

fig, axes = plt.subplots(3, 3, figsize=(12, 9), sharex=True)
for r, (metric, title, ylabel, scale) in enumerate(ROWS):
    for c, state in enumerate(STATES):
        ax = axes[r, c]
        ax.axvspan(DARK_ONSET_ZT, 24, color="0.88", zorder=0)
        for g in groups:
            d = sleep_bins_avg[(sleep_bins_avg["state"] == state) & (sleep_bins_avg["group"] == g)]
            agg = d.groupby("zt")[metric].agg(["mean", "sem"])
            ax.errorbar(agg.index, agg["mean"] * scale, yerr=agg["sem"] * scale,
                        marker="o", capsize=3, label=g)
        ax.set_title(f"{STATE_NAMES[state]}: {title}")
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4))
        if c == 0:
            ax.set_ylabel(ylabel)
        if r == 2:
            ax.set_xlabel(ZT_LABEL)
axes[0, 0].legend()
fig.tight_layout();

## 15. Whole-recording summary

The by-hour table cuts each recording into bins. This is the simpler summary: **one number per
animal and state for the whole recording**. There's no rotation and no start time involved,
because nothing depends on clock time.

For each animal and state:

- `frac_time` – epochs in the state ÷ all epochs.
- `n_bouts` – number of unbroken runs of the state in the whole recording (same `bouts()` as
  before, just run on the epoch-rate states, so lengths come back in epochs, and we multiply by
  `EPOCH_SEC` to get seconds).
- `mean_bout_sec` – average run length in seconds.
- `bouts_per_hour` – `n_bouts` ÷ recording hours, so animals with different recording lengths
  are comparable. (`n_bouts` alone grows with recording length.)
- `n_hours`, `n_days` – the recording length, kept so you can normalise some other way.

Every animal gets a row for each of `w`, `n`, `r`, even if a state never occurs there.

In [ ]:
WHOLE_METRICS = ["frac_time", "n_bouts", "mean_bout_sec", "bouts_per_hour"]


def whole_file_metrics(path):
    animal = path.name.split("_")[0]
    states = load_states(path)
    n_hours = len(states) * EPOCH_SEC / 3600
    rows = []
    for state in STATES:
        m = states == state
        n, mean_epochs = bouts(m)                    # bout length comes back in epochs
        rows.append({"mouse_id": animal, "state": state,
                     "frac_time": m.sum() / len(states),
                     "n_bouts": n, "mean_bout_sec": mean_epochs * EPOCH_SEC,
                     "bouts_per_hour": n / n_hours,
                     "n_hours": n_hours, "n_days": n_hours / 24})
    return pd.DataFrame(rows)


whole = pd.concat([whole_file_metrics(p) for p in csv_paths], ignore_index=True)
whole["group"] = whole["mouse_id"].map(animal_to_group)
whole["excluded"] = whole["mouse_id"].isin(excluded)
assert whole["group"].notna().all(), "an animal is not in any group"
whole.head(6)

**Check against the library** (optional), same as for the binned table. The library includes every animal in the folder, so we compare the animals above.

In [ ]:
from ebb.process.sleep_compute import compute_whole_file_metrics

lib_whole = compute_whole_file_metrics(SPINDLE_DIR)
m = whole.merge(lib_whole, on=["mouse_id", "state"], suffixes=("", "_lib"))
print(f"{len(m)} of {len(whole)} rows matched")
for col in ["frac_time", "n_bouts", "mean_bout_sec", "bouts_per_hour", "n_hours"]:
    print(f"{col:15s} matches library:", np.allclose(m[col], m[f"{col}_lib"], equal_nan=True))

**Prism layout.** One sheet per metric, rows = Wake / NREM / REM, columns = animals blocked by
group (blank-padded like the by-hour sheets). A long-format CSV is saved too.

In [ ]:
def write_whole_file_workbook(df, out_path):
    width = max(map(len, groups.values()))
    wb = Workbook()
    wb.remove(wb.active)
    for metric in WHOLE_METRICS:
        wide = df.pivot(index="state", columns="mouse_id", values=metric).reindex(STATES)
        ws = wb.create_sheet({"n_bouts": "bouts"}.get(metric, metric))
        layout = []
        for grp, animals in groups.items():
            block = sorted(a for a in animals if a in wide.columns)
            block += [None] * (width - len(block)) if block else []
            layout += [(grp, a) for a in block]
        for c, (grp, animal) in enumerate(layout, start=2):
            ws.cell(row=1, column=c, value=grp)
            ws.cell(row=2, column=c, value=animal)
        for r, state in enumerate(STATES, start=3):
            ws.cell(row=r, column=1, value=STATE_NAMES[state])
            for c, (_, animal) in enumerate(layout, start=2):
                val = wide.at[state, animal] if animal in wide.columns else None
                ws.cell(row=r, column=c, value=None if pd.isna(val) else float(val))
        ws.freeze_panes = "B3"
    wb.save(out_path)


WHOLE_OUT = OUT_DIR / "prism_sleep_whole_file.xlsx"
write_whole_file_workbook(whole, WHOLE_OUT)
whole.to_csv(OUT_DIR / "sleep_whole_file.csv", index=False)
print("Wrote", WHOLE_OUT)

**Plot.** Boxes show the median and quartiles for each group, with every animal's value overlaid
as a point.

In [ ]:
WHOLE_TITLES = {"frac_time": ("Time in state", "% of recording", 100),
                "n_bouts": ("Bout number", "bouts per recording", 1),
                "mean_bout_sec": ("Bout duration", "seconds", 1),
                "bouts_per_hour": ("Bout rate", "bouts per hour", 1)}

rng = np.random.default_rng(0)                    # fixed seed: same jitter every run
group_names = list(groups)
box_w = 0.8 / len(group_names)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, metric in zip(axes, WHOLE_METRICS):
    title, ylabel, scale = WHOLE_TITLES[metric]
    for gi, grp in enumerate(group_names):
        color = f"C{gi}"
        for si, state in enumerate(STATES):
            y = whole.loc[(whole["state"] == state) & (whole["group"] == grp), metric].dropna() * scale
            x = si + (gi - (len(group_names) - 1) / 2) * box_w
            ax.boxplot(y, positions=[x], widths=box_w * 0.8, showfliers=False,
                       medianprops={"color": color}, boxprops={"color": color},
                       whiskerprops={"color": color}, capprops={"color": color})
            ax.scatter(x + rng.uniform(-box_w / 4, box_w / 4, len(y)), y, s=14, color=color,
                       zorder=3, label=grp if si == 0 else None)
    ax.set_xticks(range(len(STATES)), [STATE_NAMES[s] for s in STATES])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
axes[0].legend()
fig.tight_layout();